# 시나리오 베이스 시뮬레이션

WF SCM Planning Logic — 수익 기반 용량 할당 엔진


In [ ]:
# ============================================================
# [사용 오브젝트 및 컬럼 요약]  (코드 실제 사용 기준)
# ============================================================
#
# 시뮬레이션 로직.txt 는 Palantir Foundry 온톨로지 기반
# 수익 기반 용량 할당 엔진(Revenue-Driven Allocation Engine) 설명서입니다.
#
# ── 핵심 Input 데이터셋 ──
#
# 1. WIP Lot (재공품 로트)
#    - lot_id                : 로트 ID
#    - model_id              : 모델 ID
#    - process_id            : 현재 공정 ID
#    - sequence              : 공정 순서 번호
#    - equipment_id          : 현재 설비 ID
#    - predicted_finish_time : 예상 완료 시각 (Q1 / Median / Q3 / Std)
#    - panel_quantity         : 패널 수량
#
# 2. Sales Order / Demand
#    - order_id / line_id    : 주문 / 라인 ID
#    - model_id / grouping_model / base_model : 모델 계층
#    - required_panels       : 필요 패널 수량
#    - promise_date          : 납기 약속일
#    - revenue               : 주문 수익 (할당 우선순위 기준)
#    - delivery_rank         : 납기 우선순위
#
# 3. Equipment Capacity (설비 용량)
#    - equipment_group_id    : 설비 그룹 ID
#    - process_id            : 공정 ID
#    - daily_capacity_sht    : 일일 용량 (시트 기준)
#    - site_id               : 사이트 ID
#
# 4. Model Routing (공정 라우팅)
#    - model_id              : 모델 ID
#    - process_sequence      : 공정 순서
#    - process_id            : 공정 ID
#    - std_time_per_panel    : 패널당 표준 처리 시간
#
# ── 핵심 Output 데이터셋 ──
#
# ALLOCATION_OUTPUT (할당 결과)
#    - allocation_id         : 고유 할당 ID
#    - lot_id / model_id / process_id
#    - assigned_order_id     : 할당된 주문
#    - allocated_panels      : 할당 패널 수
#    - allocation_month      : 할당 월
#    - revenue_contribution  : 수익 기여도
#
# NEW_LOTS_OUTPUT (신규 가상 로트)
#    - lot_id                : 가상 로트 ID (VL-{model}-{month}-{uuid})
#    - model_id / target_month / target_panels
#
# FAILED_ALLOCATION (할당 실패)
#    - lot_id / model_id / process_id / failure_reason / failure_status
#
# UNROUTED_MODEL_DEMAND (라우팅 없는 수요)
#    - model_id / demand_qty_ea / shortfall_month / title
#
# RUN_TRACKER (실행 추적)
#    - simulation_id / config_hash / status
#    - allocation_rows / runtime_seconds
#
# ── 핵심 설정 오브젝트 ──
#
# AllocationConstraints (할당 제약)
#    - max_lot_size_panels   : 최대 로트 크기 (패널)
#    - daily_start_capacity  : 일일 시작 용량 (로트 수)
#    - force_snapshot        : 스냅샷 강제 여부
#
# BLOCKED_EQUIPMENT_GROUPS  : 가상 로트 생성 불가 설비 그룹 목록
# EQUIPMENT_COLUMN_TO_GROUP : 설비 용량 컬럼명 → 설비 그룹명 매핑
# ============================================================


In [ ]:
# # 🏭 WF SCM Planning Logic - 수익 기반 용량 할당 엔진


```
╔═══════════════════════════════════════════════════════════════════════════════════════╗
║                                                                                       ║
║   ██╗    ██╗███████╗    ███████╗ ██████╗███╗   ███╗    ██████╗ ██╗      █████╗ ███╗  ║
║   ██║    ██║██╔════╝    ██╔════╝██╔════╝████╗ ████║    ██╔══██╗██║     ██╔══██╗████╗ ║
║   ██║ █╗ ██║█████╗      ███████╗██║     ██╔████╔██║    ██████╔╝██║     ███████║██╔██╗║
║   ██║███╗██║██╔══╝      ╚════██║██║     ██║╚██╔╝██║    ██╔═══╝ ██║     ██╔══██║██║╚█║║
║   ╚███╔███╔╝██║         ███████║╚██████╗██║ ╚═╝ ██║    ██║     ███████╗██║  ██║██║ █║║
║    ╚══╝╚══╝ ╚═╝         ╚══════╝ ╚═════╝╚═╝     ╚═╝    ╚═╝     ╚══════╝╚═╝  ╚═╝╚═╝ ╚║║
║                                                                                       ║
║                    수익 기반 용량 할당 엔진 (Revenue-Driven Allocation Engine)         ║
║                                                                                       ║
╚═══════════════════════════════════════════════════════════════════════════════════════╝
```


In [ ]:
# ---
#
# ## 📋 목차
#
# 1. [개요](#-개요)
# 2. [시스템 아키텍처](#-시스템-아키텍처)
# 3. [입력 데이터셋](#-입력-데이터셋)
# 4. [출력 데이터셋](#-출력-데이터셋)
# 5. [설정 시스템](#-설정-시스템)
# 6. [할당 알고리즘](#-할당-알고리즘)
# 7. [특수 케이스 처리](#-특수-케이스-처리)
# 8. [설정 오버라이드](#-설정-오버라이드)
# 9. [실제 예시](#-실제-예시)
# 10. [문제 해결](#-문제-해결)
#
# ---
#
# ## 🎯 개요
#
# ### 할당 엔진이란?
#
# 할당 엔진은 **제조 계획을 위한 스케줄링 시스템**입니다.


```
    ┌─────────────────────────────────────────────────────────────────────────┐
    │                                                                         │
    │   🎯 할당 엔진의 핵심 역할                                              │
    │                                                                         │
    │   ┌───────────────┐     ┌───────────────┐     ┌───────────────┐        │
    │   │  📅 언제?     │     │  🔧 어디서?   │     │  📦 얼마나?   │        │
    │   │               │     │               │     │               │        │
    │   │  각 생산 단계 │     │  어떤 설비에서│     │  새 Lot이     │        │
    │   │  언제 수행?   │     │  작업 수행?   │     │  얼마나 필요? │        │
    │   └───────────────┘     └───────────────┘     └───────────────┘        │
    │                                                                         │
    └─────────────────────────────────────────────────────────────────────────┘
```


In [ ]:
# 공장의 **항공 관제사**처럼, 모든 작업이 설비 과부하 없이 제시간에 완료되도록 조율합니다.
#
# ### 핵심 용어


```
┌──────────────────┬────────────────────────────────────────────────────────┐
│      용어        │                        설명                            │
├──────────────────┼────────────────────────────────────────────────────────┤
│  Lot (로트)      │  제조 공정을 거치는 제품 배치                          │
│  WIP             │  Work In Progress - 이미 진행 중인 기존 로트           │
│  Virtual Lot     │  수요 부족분을 채우기 위해 엔진이 생성한 가상 로트     │
│  Process Step    │  제조 순서의 단일 작업 (예: 드릴링, AOI 검사)          │
│  Equipment Group │  동일한 작업을 수행할 수 있는 설비 그룹                │
│  Target Month    │  수요가 충족되어야 하는 목표 월                        │
│  Allocation      │  특정 날짜와 설비에 로트 단계를 할당하는 것            │
│  Carryover       │  이번 달에 미충족된 수요를 다음 달로 이월              │
└──────────────────┴────────────────────────────────────────────────────────┘
```


In [ ]:
# ---
#
# ## 🏗 시스템 아키텍처
#
# ### 모듈 구조


```
allocation_engine/
│
├── 📄 revenue_driven_allocation_refactored.py   ◄── 메인 변환 & 오케스트레이션
│       │
│       │   "엔진의 두뇌 - 전체 할당 프로세스 조율"
│       │
├── 📄 config.py                                  ◄── 모든 설정 파라미터
│       │
│       │   "설정의 단일 소스 - 매직 넘버 없음"
│       │
├── 📄 models.py                                  ◄── 데이터 구조 & 열거형
│       │
│       │   "AllocationState, LotStepAllocation 등"
│       │
├── 📄 allocation_helpers.py                      ◄── 룩업 빌더 & 유틸리티
│       │
│       │   "설비 용량, 프로세스 매핑 등 구축"
│       │
├── 📄 virtual_lot_creator.py                     ◄── 가상 로트 생성
│       │
│       │   "부족분을 채우기 위한 새 로트 생성"
│       │
└── 📄 allocation_run_tracker.py                  ◄── 증분 처리 추적
        │
        │   "이미 처리된 시뮬레이션 추적"
```


In [ ]:
# ### 데이터 흐름


```
╔═══════════════════════════════════════════════════════════════════════════════════════╗
║                              📊 데이터 흐름도                                         ║
╠═══════════════════════════════════════════════════════════════════════════════════════╣
║                                                                                       ║
║   ┌─────────────────────────────────────────────────────────────────────────────┐    ║
║   │                           1️⃣  로드 단계                                      │    ║
║   │                                                                             │    ║
║   │   ┌──────────────┐      ┌──────────────────┐      ┌──────────────────┐     │    ║
║   │   │ 설정 데이터셋│ ───▶ │ 새 시뮬레이션    │ ───▶ │ 무거운 데이터셋  │     │    ║
║   │   │ 로드         │      │ 확인             │      │ 로드             │     │    ║
║   │   └──────────────┘      └────────┬─────────┘      └──────────────────┘     │    ║
║   │                                  │                                          │    ║
║   │                         [없으면 작업 중단]                                  │    ║
║   └─────────────────────────────────────────────────────────────────────────────┘    ║
║                                      │                                               ║
║                                      ▼                                               ║
║   ┌─────────────────────────────────────────────────────────────────────────────┐    ║
║   │                           2️⃣  룩업 구축                                      │    ║
║   │                                                                             │    ║
║   │   ┌────────────────────────────────────────────────────────────────────┐   │    ║
║   │   │  • 모델 메타데이터        • 설비 용량                              │   │    ║
║   │   │  • 모델 프로세스 단계     • 프로세스 → 설비 매핑                   │   │    ║
║   │   │  • 모델 우선순위          • 네거티브 제약조건                      │   │    ║
║   │   └────────────────────────────────────────────────────────────────────┘   │    ║
║   └─────────────────────────────────────────────────────────────────────────────┘    ║
║                                      │                                               ║
║                                      ▼                                               ║
║   ┌─────────────────────────────────────────────────────────────────────────────┐    ║
║   │                           3️⃣  할당 루프                                      │    ║
║   │                                                                             │    ║
║   │   ┌─────────────────────────────────────────────────────────────────────┐  │    ║
║   │   │  FOR 각 시뮬레이션:                                                 │  │    ║
║   │   │    FOR 각 월 (순서대로):                                            │  │    ║
║   │   │      FOR 각 모델 (우선순위 순):                                     │  │    ║
║   │   │                                                                     │  │    ║
║   │   │        ┌─────────────────────────────────────────────────────┐     │  │    ║
║   │   │        │ Phase 1: 기존 로트 할당 (WIP + 이전 가상 로트)      │     │  │    ║
║   │   │        │         └─▶ 완료에 가까운 로트부터 선택             │     │  │    ║
║   │   │        │         └─▶ 완료에서 먼 로트부터 할당               │     │  │    ║
║   │   │        └─────────────────────────────────────────────────────┘     │  │    ║
║   │   │                              │                                      │  │    ║
║   │   │                              ▼                                      │  │    ║
║   │   │        ┌─────────────────────────────────────────────────────┐     │  │    ║
║   │   │        │ Phase 2: 부족분을 위한 가상 로트 생성 & 할당        │     │  │    ║
║   │   │        │         └─▶ 필요한 로트 수 계산                     │     │  │    ║
║   │   │        │         └─▶ 시차별 시작일 생성                      │     │  │    ║
║   │   │        │         └─▶ 단계 순차 할당                          │     │  │    ║
║   │   │        └─────────────────────────────────────────────────────┘     │  │    ║
║   │   └─────────────────────────────────────────────────────────────────────┘  │    ║
║   └─────────────────────────────────────────────────────────────────────────────┘    ║
║                                      │                                               ║
║                                      ▼                                               ║
║   ┌─────────────────────────────────────────────────────────────────────────────┐    ║
║   │                           4️⃣  출력 단계                                      │    ║
║   │                                                                             │    ║
║   │   ┌────────────────────────────────────────────────────────────────────┐   │    ║
║   │   │  출력: 할당 결과, 새 로트, 실패한 할당, 미할당 수요, 실행 추적기  │   │    ║
║   │   └────────────────────────────────────────────────────────────────────┘   │    ║
║   └─────────────────────────────────────────────────────────────────────────────┘    ║
║                                                                                       ║
╚═══════════════════════════════════════════════════════════════════════════════════════╝
```


In [ ]:
# ---
#
# ## 📥 입력 데이터셋
#
# ### 주요 입력


```
┌──────────────────────────┬─────────────────────────────────────────────────────────┐
│       데이터셋           │                      설명                               │
├──────────────────────────┼─────────────────────────────────────────────────────────┤
│  wip_lots                │  현재 진행 중인 로트 단계 (WIP)                         │
│  net_demand              │  모델/월별 순 생산 수요                                 │
│  model_priorities        │  월별 모델 우선순위 순위                                │
│  equipment_capacity      │  설비별 일일 용량 (시트/일)                             │
│  equipment_constraints   │  특정 모델에 대해 차단된 설비                           │
│  equipment_to_process    │  프로세스 → 설비 매핑                                   │
│  model_dataset           │  모델 메타데이터 (리드타임, 고객사)                     │
│  model_unit_conversion   │  유닛/패널, 패널/시트 변환                              │  
│  planned_process_steps   │  모델별 제조 라우팅                                     │
│  simulation_config       │  시뮬레이션별 설정 오버라이드                           │
└──────────────────────────┴─────────────────────────────────────────────────────────┘
```


In [ ]:
# ### WIP 로트 스키마 (주요 컬럼)


In [ ]:
{
    "lot_id":               str,   # 고유 로트 식별자
    "model_id":             str,   # 제품 모델
    "process_id":           str,   # 현재 프로세스 단계
    "sequence":             int,   # 단계 순서 번호
    "equipment_group_id":   str,   # 이 단계의 설비 그룹
    "latest_sheet_quantity": int,  # 시트 수량
    "latest_unit_quantity": int,   # 유닛 수량
    "remaining_steps":      int,   # 남은 단계 수
    "final_work_sequence":  int    # 라우팅의 총 단계 수
}


In [ ]:
# ---
#
# ## 📤 출력 데이터셋
#
# ### 출력 개요


```
                                ┌───────────────────────┐
                                │    할당 엔진 출력     │
                                └───────────┬───────────┘
                                            │
            ┌───────────────┬───────────────┼───────────────┬───────────────┐
            │               │               │               │               │
            ▼               ▼               ▼               ▼               ▼
    ┌───────────────┐ ┌───────────┐ ┌───────────────┐ ┌───────────┐ ┌───────────┐
    │ 📋 할당 결과  │ │ 📦 새 로트│ │ ❌ 실패한 할당│ │ ⚠️ 미할당 │ │ 📊 실행   │
    │               │ │           │ │               │ │   수요    │ │   추적기  │
    │ allocation    │ │ new_lots  │ │ failed_       │ │ unrouted  │ │ run_      │
    │ _output       │ │ _created  │ │ allocations   │ │ _demand   │ │ tracker   │
    └───────────────┘ └───────────┘ └───────────────┘ └───────────┘ └───────────┘
```


In [ ]:
# ### 1️⃣ 할당 결과 (`allocation_output`)
#
# 성공적으로 할당된 모든 로트 단계를 포함합니다.


In [ ]:
ALLOCATION_OUTPUT_SCHEMA = {
    "allocation_id":           pl.Utf8,      # 고유 할당 ID
    "lot_id":                  pl.Utf8,      # 할당된 로트
    "model_id":                pl.Utf8,      # 제품 모델
    "process_id":              pl.Utf8,      # 프로세스 단계
    "sequence":                pl.Int32,     # 단계 순서
    "equipment_id":            pl.Utf8,      # 할당된 설비
    "allocated_date":          pl.Date,      # 스케줄된 날짜
    "target_month":            pl.Utf8,      # 목표 수요 월 (YYYYMM)
    "actual_completion_month": pl.Utf8,      # 실제 완료 월
    "is_delayed_completion":   pl.Boolean,   # 목표 후 완료시 True
    "model_priority":          pl.Int32,     # 우선순위 (낮을수록 높음)
    "units_produced":          pl.Int32,     # 유닛 수 (마지막 단계만)
    "delay_reasons":           pl.Utf8,      # 지연 발생 이유
    "is_new_lot":              pl.Boolean,   # 가상 로트면 True
}


In [ ]:
# ### 2️⃣ 새 로트 생성 (`new_lots_created_output`)
#
# 수요 부족분을 채우기 위해 생성된 가상 로트입니다.


In [ ]:
NEW_LOTS_OUTPUT_SCHEMA = {
    "lot_id":               pl.Utf8,   # 가상 로트 ID (VL-{model}-{month}-{uuid})
    "model_id":             pl.Utf8,
    "target_month":         pl.Utf8,
    "target_panels":        pl.Int64,
    "target_units":         pl.Int64,
    "target_lot_start_date": pl.Date,
    "simulation_id":        pl.Utf8,
    "revenue_plan_id":      pl.Utf8,
}


In [ ]:
# ### 3️⃣ 실패한 할당 (`failed_allocations_output`)
#
# 제약 조건 내에서 스케줄링할 수 없는 로트 단계입니다.


In [ ]:
FAILED_ALLOCATION_SCHEMA = {
    "lot_id":          pl.Utf8,
    "model_id":        pl.Utf8,
    "process_id":      pl.Utf8,
    "failure_reason":  pl.Utf8,    # 상세 실패 이유
    "failure_status":  pl.Utf8,    # FAILED_INSUFFICIENT_CAPACITY 등
    "days_searched":   pl.Int32,   # 시도한 날짜 수
}


In [ ]:
# ### 4️⃣ 미할당 모델 수요 (`unrouted_model_demand_output`)
#
# 수요는 있지만 제조 라우팅이 정의되지 않은 모델입니다.


In [ ]:
UNROUTED_MODEL_DEMAND_SCHEMA = {
    "model_id":              pl.Utf8,
    "demand_qty_ea":         pl.Int64,
    "shortfall_month":       pl.Utf8,
    "title":                 pl.Utf8,   # 사람이 읽을 수 있는 요약
    "remediation_suggestion": pl.Utf8,  # 취해야 할 조치 (한국어)
}


In [ ]:
# ### 5️⃣ 실행 추적기 (`run_tracker_output`)
#
# 증분 처리 상태를 추적합니다.


In [ ]:
RUN_TRACKER_SCHEMA = {
    "simulation_id":    pl.Utf8,
    "config_hash":      pl.Utf8,     # 사용된 설정의 해시
    "status":           pl.Utf8,     # SUCCESS, FAILED, SKIPPED
    "allocation_rows":  pl.Int64,
    "run_timestamp":    pl.Datetime,
}


In [ ]:
# ---
#
# ## ⚙️ 설정 시스템
#
# ### 설정 계층 구조


```
╔═══════════════════════════════════════════════════════════════════════════════════════╗
║                              ⚙️ 설정 로딩 계층                                        ║
╠═══════════════════════════════════════════════════════════════════════════════════════╣
║                                                                                       ║
║                    ┌──────────────────────────────────────┐                          ║
║                    │    1️⃣  DEFAULT_CONFIG                │                          ║
║                    │    (config.py에 하드코딩)             │                          ║
║                    └──────────────────┬───────────────────┘                          ║
║                                       │                                               ║
║                                       │  기본값 제공                                  ║
║                                       ▼                                               ║
║                    ┌──────────────────────────────────────┐                          ║
║                    │    2️⃣  simulation_config 데이터셋    │                          ║
║                    │    (시뮬레이션별 오버라이드)          │                          ║
║                    └──────────────────┬───────────────────┘                          ║
║                                       │                                               ║
║                                       │  값 오버라이드                                ║
║                                       ▼                                               ║
║                    ┌──────────────────────────────────────┐                          ║
║                    │    3️⃣  최종 AllocationConfig         │                          ║
║                    │    (각 시뮬레이션에 사용)             │                          ║
║                    └──────────────────────────────────────┘                          ║
║                                                                                       ║
║   💡 시뮬레이션별 설정이 없으면 → DEFAULT_CONFIG 사용                                ║
║                                                                                       ║
╚═══════════════════════════════════════════════════════════════════════════════════════╝
```


In [ ]:
# ### AllocationConstraints (할당 제약조건)


In [ ]:
@dataclass(frozen=True)
class AllocationConstraints:
    """
    ┌─────────────────────────────────────────────────────────────────────────────┐
    │                        📏 할당 제약조건 설정                                │
    └─────────────────────────────────────────────────────────────────────────────┘
    """
    
    # ═══════════════════════════════════════════════════════════════════════════
    # 🚀 단계 처리량 제한
    # ═══════════════════════════════════════════════════════════════════════════
    max_steps_per_lot_per_day: int = 2              # 일반 로트: 일당 최대 2단계
    max_steps_per_lot_per_day_fast_track: int = 5   # 고우선 로트: 일당 최대 5단계
    
    # ═══════════════════════════════════════════════════════════════════════════
    # ⚡ 패스트트랙 시스템
    # ═══════════════════════════════════════════════════════════════════════════
    fast_track_lots_per_day: int = 200              # 일일 패스트트랙 쿼터
    fast_track_priority_threshold: int = 10000      # 이 우선순위 이하면 패스트트랙
    
    # ═══════════════════════════════════════════════════════════════════════════
    # 🔍 검색 제한
    # ═══════════════════════════════════════════════════════════════════════════
    max_delay_days: int = 200                       # 용량 검색 최대 일수
    max_allocation_year: int = 2027                 # 이 연도 이후는 스케줄 불가
    
    # ═══════════════════════════════════════════════════════════════════════════
    # 📅 계획 기간
    # ═══════════════════════════════════════════════════════════════════════════
    start_month: str = "202602"                     # 이 월 이전은 처리하지 않음
    
    # ═══════════════════════════════════════════════════════════════════════════
    # 🏭 WIP 처리
    # ═══════════════════════════════════════════════════════════════════════════
    wip_lead_time_buffer_factor: float = 1.2        # WIP 시작일 계산용 버퍼
    use_dynamic_wip_earliest_start: bool = False    # 동적 vs 고정 시작일
    
    # ═══════════════════════════════════════════════════════════════════════════
    # 📊 수요 처리
    # ═══════════════════════════════════════════════════════════════════════════
    demand_fulfillment_buffer: float = 1.1          # 필요량의 110% 로트 선택
    
    # ═══════════════════════════════════════════════════════════════════════════
    # ⏳ 목표월까지 대기 (최종 단계용)
    # ═══════════════════════════════════════════════════════════════════════════
    hold_until_target_month_equipment_groups: FrozenSet[str] = frozenset({"입고대기"})
    enable_hold_until_target_month: bool = True
    
    # ═══════════════════════════════════════════════════════════════════════════
    # 🔄 강제 재처리
    # ═══════════════════════════════════════════════════════════════════════════
    force_snapshot: bool = False                    # 증분 로직 우회


In [ ]:
# ### DefaultValues (기본값)


In [ ]:
@dataclass(frozen=True)
class DefaultValues:
    """
    ┌─────────────────────────────────────────────────────────────────────────────┐
    │                        🔢 기본값 설정                                       │
    └─────────────────────────────────────────────────────────────────────────────┘
    """
    
    daily_capacity_sheets: int = 10            # 설비 용량 미정의시 기본값
    model_priority: int = 99999999             # 최저 우선순위 (높은 숫자)
    lead_time_days: int = 30                   # 기본 제조 리드타임
    daily_capacity_lots: int = 2               # 시차 배치용 일일 로트 수
    panels_per_lot: int = 30                   # 기본 로트 크기
    panels_per_sheet: int = 6                  # 변환 계수
    units_per_panel: int = 2400                # 변환 계수
    units_per_sheet: int = 14400               # 변환 계수
    infinite_capacity_sheets: int = 10_000_000 # 특수 설비 그룹용 무한 용량


In [ ]:
# ### 특수 설비 그룹


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 🚫 차단된 설비 그룹 (가상 로트 생성 불가)
# ═══════════════════════════════════════════════════════════════════════════════
BLOCKED_EQUIPMENT_GROUPS = frozenset({
    "D/F 박리",
    "AFVI전 세정",
})

# ═══════════════════════════════════════════════════════════════════════════════
# ♾️ 무한 용량 설비 그룹 (스케줄링 불필요)
# ═══════════════════════════════════════════════════════════════════════════════
INFINITE_CAPACITY_EQUIPMENT_GROUPS = frozenset({
    "PET PEELING",
    "적층전처리(클리닝)",
    "AOI(VRS)",
    "입고대기",
    "회로 정면",
    "휨검사",
    "BAKING(UNIT)",
    "박스분리",
    "AOI(Scan)",
    "V/M(휨검사)",
    "임피던스 측정",
    "(SOP)DEFLUX",
    "진공포장",
    "적층전처리(CZ)",
    "TP",
    "PNL 분리",
    "TNR PACKING",
    "Q/A",
})


In [ ]:
# ---
#
# ## 🔄 할당 알고리즘
#
# ### 우선순위 계층 구조


```
╔═══════════════════════════════════════════════════════════════════════════════════════╗
║                              🏆 우선순위 계층 구조                                    ║
╠═══════════════════════════════════════════════════════════════════════════════════════╣
║                                                                                       ║
║   1️⃣  월 우선순위 (순차 처리)                                                        ║
║       │                                                                               ║
║       └──▶ 월별 순서대로 처리: 202602 → 202603 → 202604 → ...                        ║
║                                                                                       ║
║   ─────────────────────────────────────────────────────────────────────────────────  ║
║                                                                                       ║
║   2️⃣  모델 우선순위 (월별)                                                           ║
║       │                                                                               ║
║       ├──▶ 낮은 숫자 = 높은 우선순위                                                 ║
║       └──▶ 우선순위 1 모델이 우선순위 2보다 먼저 할당                                 ║
║                                                                                       ║
║   ─────────────────────────────────────────────────────────────────────────────────  ║
║                                                                                       ║
║   3️⃣  로트 선택 (모델별)                                                             ║
║       │                                                                               ║
║       ├──▶ 완료에 가까운 로트부터 선택 (남은 단계 최소)                               ║
║       └──▶ 수요 × 버퍼를 충족할 만큼만 선택                                           ║
║                                                                                       ║
║   ─────────────────────────────────────────────────────────────────────────────────  ║
║                                                                                       ║
║   4️⃣  로트 할당 순서 (선택된 로트 내)                                                ║
║       │                                                                               ║
║       ├──▶ 완료에서 먼 로트부터 할당 (남은 단계 최대)                                 ║
║       └──▶ 모든 로트가 병렬로 진행되도록 보장                                         ║
║                                                                                       ║
║   ─────────────────────────────────────────────────────────────────────────────────  ║
║                                                                                       ║
║   5️⃣  단계 순서 (로트별)                                                             ║
║       │                                                                               ║
║       └──▶ 엄격히 순차적: 단계 1 완료 후에만 단계 2 가능                              ║
║                                                                                       ║
║   ─────────────────────────────────────────────────────────────────────────────────  ║
║                                                                                       ║
║   6️⃣  설비 선택 (단계별)                                                             ║
║       │                                                                               ║
║       ├──▶ 프로세스의 설비 풀 내에서 가장 여유 있는 설비                              ║
║       └──▶ 네거티브 제약조건 준수 (차단된 설비 회피)                                  ║
║                                                                                       ║
╚═══════════════════════════════════════════════════════════════════════════════════════╝
```


In [ ]:
# ### 메인 할당 루프 의사 코드


In [ ]:
def allocate_month_by_month(...):
    """
    ┌─────────────────────────────────────────────────────────────────────────────┐
    │                         메인 할당 알고리즘                                  │
    └─────────────────────────────────────────────────────────────────────────────┘
    
    FOR 각 월 (순서대로):
        FOR 각 모델 (우선순위 순):
            
            1. 총 수요 계산
               ┌────────────────────────────────────────┐
               │ 총 수요 = 기본 수요 + 이전 월 이월분   │
               └────────────────────────────────────────┘
            
            2. Phase 1: 기존 로트 할당
               ┌────────────────────────────────────────┐
               │ • 완료에 가까운 로트 선택              │
               │ • 힙 사용하여 완료에서 먼 것부터 할당  │
               │ • units_allocated 추적                 │
               └────────────────────────────────────────┘
            
            3. Phase 2: 부족분용 가상 로트 생성
               ┌────────────────────────────────────────┐
               │ IF 수요 > 할당된 수량:                 │
               │   • 가상 로트 생성                     │
               │   • 시차별 시작일 계산                 │
               │   • 모든 단계 순차 할당                │
               └────────────────────────────────────────┘
            
            4. 이월 계산
               ┌────────────────────────────────────────┐
               │ 이월 = 할당 불가능했던 수요            │
               │       (다음 월로 이월)                 │
               └────────────────────────────────────────┘
    """


In [ ]:
# ### 설비 선택 로직


In [ ]:
def find_least_loaded_equipment_for_process(process_id, target_date, ...):
    """
    ┌─────────────────────────────────────────────────────────────────────────────┐
    │                         설비 선택 알고리즘                                  │
    └─────────────────────────────────────────────────────────────────────────────┘
    
    1. 이 process_id를 처리할 수 있는 모든 설비 조회
    
    2. 이 모델에 대해 차단된 설비 필터링 (네거티브 제약조건)
    
    3. target_date에 각 설비의 용량 확인
    
    4. 현재 사용량이 가장 낮은 설비 선택
    
    5. 반환: (equipment_id, 잔여_용량, 차단된_설비_정보)
    """


In [ ]:
# ### 패스트트랙 시스템


```
┌─────────────────────────────────────────────────────────────────────────────────────┐
│                              ⚡ 패스트트랙 시스템                                   │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│   고우선순위 모델은 하루에 더 많은 단계를 처리할 수 있습니다:                       │
│                                                                                     │
│   ┌─────────────────────────────────────────────────────────────────────────────┐  │
│   │                                                                             │  │
│   │   일반 로트:     ██░░░░░░░░  일당 최대 2단계                               │  │
│   │                                                                             │  │
│   │   패스트트랙:    █████░░░░░  일당 최대 5단계                               │  │
│   │                                                                             │  │
│   └─────────────────────────────────────────────────────────────────────────────┘  │
│                                                                                     │
│   📋 패스트트랙 조건:                                                              │
│                                                                                     │
│   • 모델 우선순위 ≤ 10,000 (fast_track_priority_threshold)                         │
│   • 해당 일의 패스트트랙 쿼터 미소진 (일당 200개)                                  │
│   • 한 번 패스트트랙으로 지정되면 해당 로트의 모든 단계에 적용                     │
│                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────┘
```


In [ ]:
# ---
#
# ## 🔧 특수 케이스 처리
#
# ### 1️⃣ 목표월까지 대기 (최종 단계)


```
┌─────────────────────────────────────────────────────────────────────────────────────┐
│                         ⏳ 목표월까지 대기 로직                                     │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│   특정 설비 그룹 (예: "입고대기")의 최종 단계는 목표 월까지 대기합니다:             │
│                                                                                     │
│   예시: 목표 월 = 202512 (2025년 12월)                                             │
│                                                                                     │
│       단계 1: DRILL    ──▶  11월 15일 완료                                         │
│       단계 2: AOI      ──▶  11월 16일 완료                                         │
│       단계 3: 입고대기 ──▶  12월 1일에 할당 (목표 월 첫날)                         │
│                              ▲                                                      │
│                              │                                                      │
│                        조기 납품 방지                                               │
│                                                                                     │
│   💡 이 설비 그룹들은 무한 용량으로 설정되어 용량 검색을 건너뜁니다                 │
│                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────┘
```


In [ ]:
# ### 2️⃣ 그룹핑 모델 제약조건 매칭


```
┌─────────────────────────────────────────────────────────────────────────────────────┐
│                         🔗 그룹핑 모델 제약조건                                     │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│   동일한 grouping_model을 가진 모델들은 설비 제약조건을 공유합니다:                 │
│                                                                                     │
│   ┌─────────────────────────────────────────────────────────────────────────────┐  │
│   │                                                                             │  │
│   │   모델 A1  ──┐                                                              │  │
│   │              │                                                              │  │
│   │   모델 A2  ──┼──▶  grouping_model = "ModelA"                               │  │
│   │              │                                                              │  │
│   │   모델 A3  ──┘                                                              │  │
│   │                                                                             │  │
│   │   제약조건: (ModelA, DRILL) → DRILL-05 차단                                 │  │
│   │                                                                             │  │
│   │   결과: A1, A2, A3 모두 DRILL 공정에서 DRILL-05 사용 불가                   │  │
│   │                                                                             │  │
│   └─────────────────────────────────────────────────────────────────────────────┘  │
│                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────┘
```


In [ ]:
# ### 3️⃣ 차단된 설비 경로


```
┌─────────────────────────────────────────────────────────────────────────────────────┐
│                         🚫 차단된 설비 경로 감지                                    │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│   특정 프로세스 단계의 모든 설비가 해당 모델에 대해 차단된 경우:                    │
│                                                                                     │
│   예시:                                                                             │
│   ┌─────────────────────────────────────────────────────────────────────────────┐  │
│   │                                                                             │  │
│   │   프로세스: DRILL                                                           │  │
│   │   사용 가능 설비: [DRILL-01, DRILL-02, DRILL-03]                            │  │
│   │                                                                             │  │
│   │   모델 X 제약조건:                                                          │  │
│   │     • DRILL-01 ❌ 차단                                                      │  │
│   │     • DRILL-02 ❌ 차단                                                      │  │
│   │     • DRILL-03 ❌ 차단                                                      │  │
│   │                                                                             │  │
│   │   결과: 모든 설비 차단 → 즉시 실패 처리                                     │  │
│   │   실패 이유: "All equipment for process DRILL is blocked for model"         │  │
│   │                                                                             │  │
│   └─────────────────────────────────────────────────────────────────────────────┘  │
│                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────┘
```


In [ ]:
# ### 4️⃣ 수요 이월 로직


```
╔═══════════════════════════════════════════════════════════════════════════════════════╗
║                         📊 수요 이월 로직 (중요!)                                     ║
╠═══════════════════════════════════════════════════════════════════════════════════════╣
║                                                                                       ║
║   ⚠️  이월은 "할당된" 유닛 기준이지, "정시 완료" 기준이 아닙니다!                    ║
║                                                                                       ║
║   ┌─────────────────────────────────────────────────────────────────────────────┐    ║
║   │                                                                             │    ║
║   │   로트가 할당(스케줄)되면, 해당 수요는 충족된 것으로 간주                    │    ║
║   │                                                                             │    ║
║   │   로트가 지연 완료되더라도, 원래 수요에 대해 카운트됨                        │    ║
║   │                                                                             │    ║
║   │   이렇게 하면 이미 커버된 수요에 대해 중복 로트 생성 방지                    │    ║
║   │                                                                             │    ║
║   └─────────────────────────────────────────────────────────────────────────────┘    ║
║                                                                                       ║
║   공식:                                                                               ║
║   ┌─────────────────────────────────────────────────────────────────────────────┐    ║
║   │                                                                             │    ║
║   │   이월 = max(0, 총_수요 - 할당된_유닛)                                      │    ║
║   │                                                                             │    ║
║   │   ❌ 아님: max(0, 총_수요 - 정시_완료_유닛)                                 │    ║
║   │                                                                             │    ║
║   └─────────────────────────────────────────────────────────────────────────────┘    ║
║                                                                                       ║
╚═══════════════════════════════════════════════════════════════════════════════════════╝
```


In [ ]:
# ### 5️⃣ 미할당 모델 처리


```
┌─────────────────────────────────────────────────────────────────────────────────────┐
│                         ⚠️ 미할당 모델 처리                                         │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│   수요는 있지만 프로세스 라우팅이 없는 모델은 별도로 추적됩니다:                    │
│                                                                                     │
│   ┌─────────────────────────────────────────────────────────────────────────────┐  │
│   │                                                                             │  │
│   │   IF 모델_프로세스_단계 없음:                                               │  │
│   │       • 가상 로트 생성 불가                                                 │  │
│   │       • unrouted_model_demand_output에 기록                                 │  │
│   │       • 최종 사용자를 위한 한국어 메시지 생성                               │  │
│   │                                                                             │  │
│   │   remediation_suggestion:                                                   │  │
│   │   "이 모델의 제조 공정을 정의하려면 공정 엔지니어링에 문의하십시오."         │  │
│   │                                                                             │  │
│   └─────────────────────────────────────────────────────────────────────────────┘  │
│                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────┘
```


In [ ]:
# ### 6️⃣ 증분 처리


```
┌─────────────────────────────────────────────────────────────────────────────────────┐
│                         🔄 증분 처리 로직                                           │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│   엔진은 중복 작업을 피하기 위해 처리된 시뮬레이션을 추적합니다:                    │
│                                                                                     │
│   ┌─────────────────────────────────────────────────────────────────────────────┐  │
│   │                                                                             │  │
│   │   각 시뮬레이션은 (simulation_id, config_hash)로 식별됩니다                 │  │
│   │                                                                             │  │
│   │   처리 대상:                                                                │  │
│   │     • 새로운 시뮬레이션                                                     │  │
│   │     • 설정이 변경된 시뮬레이션                                              │  │
│   │                                                                             │  │
│   │   처리할 것이 없으면 → 작업 즉시 중단                                       │  │
│   │                                                                             │  │
│   └─────────────────────────────────────────────────────────────────────────────┘  │
│                                                                                     │
│   💡 이렇게 하면 변경되지 않은 시뮬레이션의 처리 시간이 크게 단축됩니다             │
│                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────┘
```


In [ ]:
# ---
#
# ## 🎛 설정 오버라이드
#
# ### 시뮬레이션별 설정
#
# `simulation_config` 데이터셋을 통해 시뮬레이션별로 기본값을 오버라이드할 수 있습니다:


```
┌────────────────────────────┬────────────┬───────────────────────────────────────┐
│          컬럼              │    타입    │                 설명                  │
├────────────────────────────┼────────────┼───────────────────────────────────────┤
│  primary_key_              │  string    │  시뮬레이션 ID                        │
│  effective_start_date      │  date      │  시뮬레이션 시작일                    │
│  start_month               │  int       │  처리 시작 월 (예: 202602)            │
│  max_delay_days            │  int       │  용량 검색 최대 일수                  │
│  demand_fulfillment_buffer │  float     │  버퍼 승수                            │
│  force_snapshot            │  bool      │  전체 재처리 강제                     │
│  equip_capacity_{name}     │  int       │  설비 그룹 용량 오버라이드            │
└────────────────────────────┴────────────┴───────────────────────────────────────┘
```


In [ ]:
# ### 설비 용량 오버라이드
#
# 컬럼 이름이 설비 그룹 이름에 매핑됩니다:


In [ ]:
EQUIPMENT_COLUMN_TO_GROUP_NAME = {
    "equip_capacity_aoi_scan":   "AOI(Scan)",
    "equip_capacity_ldi_노광":   "LDI 노광",
    "equip_capacity_입고대기":   "입고대기",
    "equip_capacity3d측정기":    "3D측정기",
    "equip_capacity_baking_unit": "BAKING(UNIT)",
    # ... 등
}


In [ ]:
# ### 강제 스냅샷 모드
#
# 전체 재처리를 강제하려면 (증분 로직 우회):


In [ ]:
# 옵션 1: simulation_config 데이터셋에서 설정
force_snapshot = True

# 옵션 2: DEFAULT_CONFIG에서 설정 (모든 시뮬레이션에 영향)
class AllocationConstraints:
    force_snapshot: bool = True


In [ ]:
# ---
#
# ## 📝 실제 예시
#
# ### 예시 1: 기본 할당 흐름


```
╔═══════════════════════════════════════════════════════════════════════════════════════╗
║   시나리오: 모델 A - 수요 100 유닛, WIP 50 유닛                                       ║
╠═══════════════════════════════════════════════════════════════════════════════════════╣
║                                                                                       ║
║   📥 입력:                                                                            ║
║     • 수요: 100 유닛 (2025년 11월)                                                    ║
║     • WIP: 1 로트, 50 유닛, 2단계 남음                                                ║
║     • 설비: DRILL (100 시트/일), AOI (100 시트/일)                                    ║
║                                                                                       ║
║   ═══════════════════════════════════════════════════════════════════════════════    ║
║                                                                                       ║
║   🏭 PHASE 1 - WIP 할당:                                                             ║
║                                                                                       ║
║     11월 1일:                                                                         ║
║     ┌─────────────────────────────────────────────────────────────────────────┐      ║
║     │ 로트: WIP-A-001 (50 유닛, 10 시트)                                      │      ║
║     │                                                                         │      ║
║     │ 단계 1: DRILL-01 (10/100 용량)  ✓ 할당 완료                            │      ║
║     │ 단계 2: AOI-01   (10/100 용량)  ✓ 할당 완료                            │      ║
║     │                                                                         │      ║
║     │ 결과: 50 유닛 충족                                                     │      ║
║     └─────────────────────────────────────────────────────────────────────────┘      ║
║                                                                                       ║
║     부족분: 100 - 50 = 50 유닛                                                       ║
║                                                                                       ║
║   ═══════════════════════════════════════════════════════════════════════════════    ║
║                                                                                       ║
║   📦 PHASE 2 - 가상 로트 생성:                                                       ║
║                                                                                       ║
║     ┌─────────────────────────────────────────────────────────────────────────┐      ║
║     │ 생성됨: VL-A-202511-abc1                                                │      ║
║     │ 목표: 50 유닛 (9 시트)                                                  │      ║
║     │ 시작일: 11월 2일                                                        │      ║
║     └─────────────────────────────────────────────────────────────────────────┘      ║
║                                                                                       ║
║   ═══════════════════════════════════════════════════════════════════════════════    ║
║                                                                                       ║
║   🔧 PHASE 2 - 가상 로트 할당:                                                       ║
║                                                                                       ║
║     11월 2일:                                                                         ║
║     ┌─────────────────────────────────────────────────────────────────────────┐      ║
║     │ 로트: VL-A-202511-abc1                                                  │      ║
║     │                                                                         │      ║
║     │ 단계 1: DRILL-01 (9/100)  ✓ 할당 완료                                  │      ║
║     │ 단계 2: AOI-01   (9/100)  ✓ 할당 완료                                  │      ║
║     │                                                                         │      ║
║     │ 결과: 50 유닛 충족                                                     │      ║
║     └─────────────────────────────────────────────────────────────────────────┘      ║
║                                                                                       ║
║   ═══════════════════════════════════════════════════════════════════════════════    ║
║                                                                                       ║
║   📊 최종 결과:                                                                       ║
║     • 총 충족: 100 유닛   ✓                                                          ║
║     • 부족분: 0 유닛                                                                 ║
║     • 생성된 가상 로트: 1개                                                          ║
║                                                                                       ║
╚═══════════════════════════════════════════════════════════════════════════════════════╝
```


In [ ]:
# ### 예시 2: 용량 제약으로 인한 지연


```
╔═══════════════════════════════════════════════════════════════════════════════════════╗
║   시나리오: 설비 용량 근접으로 인한 지연 발생                                         ║
╠═══════════════════════════════════════════════════════════════════════════════════════╣
║                                                                                       ║
║   📊 설비 상태 (11월 28일):                                                          ║
║                                                                                       ║
║     DRILL-01: [95/100 시트]  █████████░  5 시트 여유                                 ║
║     DRILL-02: [92/100 시트]  █████████░  8 시트 여유                                 ║
║                                                                                       ║
║   ═══════════════════════════════════════════════════════════════════════════════    ║
║                                                                                       ║
║   📦 로트 필요량: 10 시트                                                            ║
║                                                                                       ║
║   ═══════════════════════════════════════════════════════════════════════════════    ║
║                                                                                       ║
║   🔍 할당 시도:                                                                       ║
║                                                                                       ║
║     11월 28일:                                                                        ║
║       DRILL-01: 95 + 10 = 105 > 100  ❌ 용량 부족                                    ║
║       DRILL-02: 92 + 10 = 102 > 100  ❌ 용량 부족                                    ║
║                                                                                       ║
║       → 11월 29일로 지연                                                              ║
║                                                                                       ║
║     11월 29일:                                                                        ║
║       DRILL-01: [40/100]  ████░░░░░░  ✓ 할당 성공                                   ║
║                                                                                       ║
║   ═══════════════════════════════════════════════════════════════════════════════    ║
║                                                                                       ║
║   📝 기록된 지연 이유:                                                               ║
║                                                                                       ║
║     "Nov 28: DRILL-01 full (95/100 sheets, needed 10);                               ║
║      Nov 28: DRILL-02 full (92/100 sheets, needed 10)"                               ║
║                                                                                       ║
╚═══════════════════════════════════════════════════════════════════════════════════════╝
```


In [ ]:
# ### 예시 3: 월별 우선순위 변경


```
╔═══════════════════════════════════════════════════════════════════════════════════════╗
║   시나리오: 모델 우선순위가 월별로 변경됨                                             ║
╠═══════════════════════════════════════════════════════════════════════════════════════╣
║                                                                                       ║
║   📊 우선순위 테이블:                                                                ║
║                                                                                       ║
║   ┌──────────┬────────────────┬────────────────┐                                     ║
║   │   모델   │  11월 우선순위 │  12월 우선순위 │                                     ║
║   ├──────────┼────────────────┼────────────────┤                                     ║
║   │ 모델 A   │       1        │       2        │                                     ║
║   │ 모델 B   │       2        │       1        │  ◀── 우선순위 역전!                 ║
║   └──────────┴────────────────┴────────────────┘                                     ║
║                                                                                       ║
║   ═══════════════════════════════════════════════════════════════════════════════    ║
║                                                                                       ║
║   🔄 처리 순서:                                                                       ║
║                                                                                       ║
║     11월:                                                                             ║
║       1️⃣  모델 A (우선순위 1) ◀── 먼저 할당, 용량 우선 확보                          ║
║       2️⃣  모델 B (우선순위 2) ◀── 나중 할당, 용량 제약 가능성                        ║
║                                                                                       ║
║     12월:                                                                             ║
║       1️⃣  모델 B (우선순위 1) ◀── 이번 달에는 먼저 할당!                             ║
║       2️⃣  모델 A (우선순위 2) ◀── 이번 달에는 나중 할당                              ║
║                                                                                       ║
║   💡 우선순위는 고정이 아닌 월별로 동적입니다!                                        ║
║                                                                                       ║
╚═══════════════════════════════════════════════════════════════════════════════════════╝
```


In [ ]:
# ### 예시 4: 연쇄 지연 효과


```
╔═══════════════════════════════════════════════════════════════════════════════════════╗
║   시나리오: 한 설비의 과부하가 여러 주문에 미치는 영향                                ║
╠═══════════════════════════════════════════════════════════════════════════════════════╣
║                                                                                       ║
║   📊 설정:                                                                            ║
║     • 설비 X: 일일 용량 200 시트                                                     ║
║     • 5개 주문이 모두 같은 날 시작 희망                                              ║
║                                                                                       ║
║   ┌─────────────┬────────────┬─────────────┐                                         ║
║   │    주문     │   시트 수  │   우선순위  │                                         ║
║   ├─────────────┼────────────┼─────────────┤                                         ║
║   │   주문 A    │    80      │      1      │                                         ║
║   │   주문 B    │    70      │      2      │                                         ║
║   │   주문 C    │    90      │      3      │                                         ║
║   │   주문 D    │    60      │      4      │                                         ║
║   │   주문 E    │    50      │      5      │                                         ║
║   ├─────────────┼────────────┼─────────────┤                                         ║
║   │    합계     │   350      │ (용량 175%) │                                         ║
║   └─────────────┴────────────┴─────────────┘                                         ║
║                                                                                       ║
║   ═══════════════════════════════════════════════════════════════════════════════    ║
║                                                                                       ║
║   🎬 스케줄링 진행 과정:                                                             ║
║                                                                                       ║
║   12/04 ┌──────────────────────────────────────────────────────┐                     ║
║         │ ▓▓▓▓▓A(80)▓▓▓▓▓▓▒▒▒B(70)▒▒▒▒░░░░░░░░░░░░░░░░░░░░░░  │ 150/200            ║
║         │ ░C(90)?→ 초과!                                       │                     ║
║         └──────────────────────────────────────────────────────┘                     ║
║                 │                                                                     ║
║                 └─▶ 주문 C, D, E 밀림                                                ║
║                                                                                       ║
║   12/05 ┌──────────────────────────────────────────────────────┐                     ║
║         │ ▓▓▓▓▓▓▓C(90)▓▓▓▓▓▓▒▒▒▒D(60)▒▒▒░░░░░░░░░░░░░░░░░░░░  │ 150/200            ║
║         │ ░E(50)?→ 초과!                                       │                     ║
║         └──────────────────────────────────────────────────────┘                     ║
║                 │                                                                     ║
║                 └─▶ 주문 E만 밀림                                                    ║
║                                                                                       ║
║   12/06 ┌──────────────────────────────────────────────────────┐                     ║
║         │ ▓▓▓E(50)▓░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░  │  50/200            ║
║         └──────────────────────────────────────────────────────┘                     ║
║                                                                                       ║
║   ═══════════════════════════════════════════════════════════════════════════════    ║
║                                                                                       ║
║   📉 지연 영향도:                                                                    ║
║                                                                                       ║
║       A ━━━ 정시 (0일 지연)                                                          ║
║       B ━━━ 정시 (0일 지연)                                                          ║
║       C ━━━━━━━ 1일 지연 ⚠️                                                          ║
║       D ━━━━━━━ 1일 지연 ⚠️                                                          ║
║       E ━━━━━━━━━━ 2일 지연 🚨                                                       ║
║                                                                                       ║
╚═══════════════════════════════════════════════════════════════════════════════════════╝
```


In [ ]:
# ---
#
# ## 🔍 문제 해결
#
# ### 일반적인 문제


```
┌─────────────────────────────────┬──────────────────────────────┬───────────────────────────────┐
│            문제                 │          가능한 원인         │            해결책             │
├─────────────────────────────────┼──────────────────────────────┼───────────────────────────────┤
│ 모든 로트 할당 실패             │ 프로세스에 설비 매핑 없음    │ equipment_to_process 확인     │
│                                 │                              │                               │
│ 예상치 못한 이월 발생           │ 설비 용량이 너무 낮음        │ equipment_capacity 검토       │
│                                 │                              │                               │
│ 시뮬레이션이 처리되지 않음      │ 동일 설정으로 이미 처리됨    │ 설정 변경 또는                │
│                                 │                              │ force_snapshot=True 설정      │
│                                 │                              │                               │
│ 가상 로트가 생성되지 않음       │ 모델에 라우팅 정의 없음      │ planned_process_steps 확인    │
│                                 │                              │                               │
│ 특정 설비만 계속 사용됨         │ 다른 설비에 네거티브 제약    │ equipment_constraints 확인    │
└─────────────────────────────────┴──────────────────────────────┴───────────────────────────────┘
```


In [ ]:
# ### 디버깅 팁


```
┌─────────────────────────────────────────────────────────────────────────────────────┐
│                              🔧 디버깅 체크리스트                                   │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│   1️⃣  failed_allocations_output 확인                                               │
│       → 상세한 실패 이유 확인 가능                                                  │
│                                                                                     │
│   2️⃣  delay_reasons 컬럼 검토                                                       │
│       → 일별 용량 검색 과정 확인                                                    │
│                                                                                     │
│   3️⃣  run_tracker의 config_hash 비교                                               │
│       → 설정 변경 여부 확인                                                         │
│                                                                                     │
│   4️⃣  force_snapshot=True 설정                                                     │
│       → 깨끗한 재처리 강제                                                          │
│                                                                                     │
│   5️⃣  unrouted_model_demand_output 확인                                            │
│       → 라우팅 없는 모델 식별                                                       │
│                                                                                     │
│   6️⃣  equipment_to_process 데이터셋 검증                                           │
│       → process_id → equipment_id 매핑 확인                                         │
│                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────┘
```


In [ ]:
# ---
#
# ## 📈 성능 고려사항
#
# ### 리소스 설정
#
# 메인 변환은 증가된 리소스로 실행됩니다:


In [ ]:
.with_resources(cpu_cores=8, memory_gb=64)


In [ ]:
# ### 증분 처리의 이점


```
┌─────────────────────────────────────────────────────────────────────────────────────┐
│                              🚀 증분 처리 최적화                                    │
├─────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                     │
│   처리할 새 시뮬레이션이 없으면 → 작업 즉시 중단                                    │
│                                                                                     │
│   ┌─────────────────────────────────────────────────────────────────────────────┐  │
│   │                                                                             │  │
│   │   첫 실행:      ████████████████████████████████████  (전체 처리)          │  │
│   │                                                                             │  │
│   │   후속 실행     ██░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░  (새것만 처리)         │
│   │   (변경 없음):                                                              │  │
│   │                                                                             │  │
│   │   후속 실행     █████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░  (변경된 것만)        │
│   │   (일부 변경):                                                              │  │
│   │                                                                             │  │
│   └─────────────────────────────────────────────────────────────────────────────┘  │
│                                                                                     │
│   💡 처리 시간이 크게 단축됩니다!                                                   │
│                                                                                     │
└─────────────────────────────────────────────────────────────────────────────────────┘
```


In [ ]:
# ---
#
# ## 📞 문의
#
# 이 할당 엔진에 대한 질문이나 문제가 있으면 SCM Planning 팀에 문의하세요.
#
# ---


```
╔═══════════════════════════════════════════════════════════════════════════════════════╗
║                                                                                       ║
║   🎯 할당 엔진의 핵심 가치                                                           ║
║                                                                                       ║
║   ┌─────────────────────────────────────────────────────────────────────────────┐    ║
║   │                                                                             │    ║
║   │   💡 가시성  ───▶  더 나은 계획                                             │    ║
║   │                                                                             │    ║
║   │   📊 데이터  ───▶  현명한 의사결정                                          │    ║
║   │                                                                             │    ║
║   │   ⚡ 최적화  ───▶  효율적 운영                                              │    ║
║   │                                                                             │    ║
║   │   🎯 정확성  ───▶  고객 신뢰                                                │    ║
║   │                                                                             │    ║
║   └─────────────────────────────────────────────────────────────────────────────┘    ║
║                                                                                       ║
║   "모든 것이 맞아떨어지길 바란다"                                                    ║
║           ↓                                                                           ║
║   "무엇이 가능하고 불가능한지 정확히 안다"                                           ║
║                                                                                       ║
╚═══════════════════════════════════════════════════════════════════════════════════════╝
```
